# Get started with Kumo Tabular

Kumo Tabular is a foundation model for a single table: it fills in a column from the other columns, with no feature engineering and no model trained for each task.
You give it rows where the answer is known and rows where it is not, and it predicts the missing answers from the known ones.

This notebook deploys the model in your workspace, then asks three questions of Microsoft's AdventureWorks products table: which color a product has, what its list price is, and the color question again as a predictive query.
Each prediction is checked against the value the table records, and compared with a simple rule, such as always guessing the most common color.

## How Kumo Tabular works

> **1. The input is one table.**

The table holds numerical and categorical columns, such as a product's weight and its size.
The model reads no related tables and no time windows, so it needs no graph: rows in, answers out.

> **2. Rows where the answer is known are the context.**

The rows where the target column is filled in are the context examples.
The rows you want answered are the rows to score, and have the target column left out.

> **3. A prediction is classification or regression.**

A categorical target, such as a color, is a classification and returns a predicted class with a probability for each class.
A numerical target, such as a price, is a regression and returns a number.

There is no training step: for each request, the client sends the context rows with the rows to score, and the model predicts from them.

## Before you start

You need:

- the Kumo Tabular model, in Unity Catalog or installed from Databricks Marketplace;
- permission to create a Model Serving endpoint, in a region with GPU Model Serving;
- a catalog and schema where you can create tables;
- serverless compute with environment 4 or later, or Databricks Runtime 17.3 LTS or later.

The first deployment of the endpoint takes 20 to 40 minutes, then the rest of the notebook runs in a few minutes.
The endpoint bills for GPU time while it runs; it scales to zero when idle, and the last step shows how to delete it.

Run each cell in order with Shift+Enter.

**Let's get started!**

## Step 1. Install the client

Install the client, the `structured-data-models-client` package, with its Databricks extras, then restart Python.
pip may print a red message about `protobuf` versions; it does not affect this notebook.

In [ ]:
%pip install --quiet "structured-data-models-client[databricks,databricks-serving]"
dbutils.library.restartPython()

## Step 2. Set your parameters

Set these names, then run the cell.
`MODEL_NAME` is the Kumo Tabular model, in the catalog you installed it into, and `CATALOG` and `SCHEMA` are where the sample table goes.

In [ ]:
from datetime import timedelta

import pandas as pd
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
    ServingModelWorkloadType,
)
from sdm_client import (
    KumoTabular,
    PredictiveQuery,
    Request,
    Schema,
    SDMClient,
)
from sdm_connectors import quote_ident

# The model, as <catalog>.<schema>.<model>, and its version.
MODEL_NAME = 'kumo_tabular.models.kumo_tabular'
MODEL_VERSION = '1'
ENDPOINT_NAME = 'kumo-tabular'
# Where the sample table goes: a catalog and schema you can create tables in.
CATALOG = 'main'
SCHEMA = 'kumo_tabular_quickstart'
TABLE = 'adventureworks_products'


def quoted(*parts):
    # A Unity Catalog name, each part quoted for SQL.
    return '.'.join(quote_ident(part, char='`') for part in parts)


PRODUCTS = quoted(CATALOG, SCHEMA, TABLE)

## Step 3. Deploy the model

The next cell creates a Model Serving endpoint for the model in your workspace, on one GPU (`GPU_MEDIUM`, size Small) that scales to zero when idle.
It does not wait for the deployment; Step 6 does.
If the endpoint already exists, the cell checks that it serves this model.
You can also create it with **Serve this model** on the model's page in Catalog Explorer.

In [ ]:
workspace = WorkspaceClient()
try:
    endpoint = workspace.serving_endpoints.get(ENDPOINT_NAME)
except NotFound:
    workspace.serving_endpoints.create(
        name=ENDPOINT_NAME,
        config=EndpointCoreConfigInput(
            name=ENDPOINT_NAME,
            served_entities=[
                ServedEntityInput(
                    entity_name=MODEL_NAME,
                    entity_version=MODEL_VERSION,
                    workload_type=ServingModelWorkloadType.GPU_MEDIUM,
                    workload_size='Small',
                    scale_to_zero_enabled=True,
                )
            ],
        ),
    )
    print(
        f'Creating the endpoint {ENDPOINT_NAME}; this takes 20 to 40 minutes.'
    )
else:
    config = endpoint.config or endpoint.pending_config
    served = {
        (entity.entity_name.lower(), entity.entity_version)
        for entity in config.served_entities
    }
    if served != {(MODEL_NAME.lower(), MODEL_VERSION)}:
        raise ValueError(
            f'The endpoint {ENDPOINT_NAME} already exists and serves {sorted(served)}, '
            f'not {MODEL_NAME} version {MODEL_VERSION}. Set ENDPOINT_NAME to another name.'
        )
    print(f'The endpoint {ENDPOINT_NAME} already exists and serves this model.')

## Step 4. Load the sample data

The next cell downloads the AdventureWorks products from the [Microsoft SQL Server samples repository](https://github.com/microsoft/sql-server-samples/tree/master/samples/databases/adventure-works/oltp-install-script) at a fixed commit and saves them as a table in your schema.
It needs internet access, and skips the download if the table already exists with the expected columns.

> Each user is responsible for checking the content of datasets and the applicable licenses and determining if suitable for the intended use.
> The repository's samples are provided under the [MIT License](https://github.com/microsoft/sql-server-samples/blob/master/license.txt).

In [ ]:
URL = (
    'https://raw.githubusercontent.com/microsoft/sql-server-samples/'
    'b47eadc85277f72843647223cde53aa1b85ffcb8/samples/databases/'
    'adventure-works/oltp-install-script/Product.csv'
)
COLUMNS = (
    'ProductID Name ProductNumber MakeFlag FinishedGoodsFlag Color '
    'SafetyStockLevel ReorderPoint StandardCost ListPrice Size '
    'SizeUnitMeasureCode WeightUnitMeasureCode Weight DaysToManufacture '
    'ProductLine Class Style ProductSubcategoryID ProductModelID '
    'SellStartDate SellEndDate DiscontinuedDate rowguid ModifiedDate'
).split()

if spark.catalog.tableExists(PRODUCTS):
    if spark.table(PRODUCTS).columns != COLUMNS:
        raise ValueError(
            f'{PRODUCTS} already exists with other columns. '
            'Drop it, or set SCHEMA to another schema.'
        )
    print(f'The sample table is already in {CATALOG}.{SCHEMA}.')
else:
    frame = pd.read_csv(
        URL,
        sep='\t',
        header=None,
        names=COLUMNS,
        dtype_backend='numpy_nullable',
    )
    for column in frame.columns:
        if column.endswith('Date'):
            frame[column] = pd.to_datetime(frame[column])
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {quoted(CATALOG, SCHEMA)}')
    spark.createDataFrame(frame).write.mode('ignore').saveAsTable(PRODUCTS)
    print(f'Saved the sample table to {CATALOG}.{SCHEMA}.')

## Step 5. Read the table and choose the features

The model reads one table, so the table is read into the notebook: unlike the relational model, which samples linked tables where they are, there is nothing to sample here.
A table small enough to score is simply read.

The model reads numerical and categorical columns.
The cell leaves out the rest: the product name, product number and `rowguid`, which name a row rather than describe it, and the dates, which are not numbers or categories.
It also leaves out `ProductID`, a row identifier, which Example 3 adds back as the key of a predictive query.

In [ ]:
products = spark.table(PRODUCTS).toPandas()
LEFT_OUT = [
    'ProductID',
    'rowguid',
    'ModifiedDate',
    'Name',
    'ProductNumber',
    'SellStartDate',
    'SellEndDate',
    'DiscontinuedDate',
]
features = products.drop(columns=LEFT_OUT)
print(f'{len(products)} products; features: {", ".join(features.columns)}')
products.head()

## Step 6. Connect to the model

The next cell waits until the endpoint is ready, then connects to it.
If the endpoint has scaled to zero, the first prediction takes a few extra minutes while it starts.

In [ ]:
endpoint = workspace.serving_endpoints.wait_get_serving_endpoint_not_updating(
    ENDPOINT_NAME, timeout=timedelta(hours=1)
)
ready = endpoint.state.ready.value if endpoint.state.ready else 'unknown'
if ready != 'READY':
    raise RuntimeError(
        f'The endpoint {ENDPOINT_NAME} is {ready}. '
        'Open it under Serving to see why, then run this cell again.'
    )
# Allow 15 minutes per request, for an endpoint that has to scale up from zero.
client = SDMClient.for_databricks_serving(ENDPOINT_NAME, timeout=900)
model = KumoTabular()
print(f'Connected to {ENDPOINT_NAME}.')

## Step 7. Make predictions

Each prediction takes two calls:

1. `Request(context=..., predict=..., target=..., task=...)` holds the labelled rows, the rows to score, the column to predict, and the kind of question.
2. `client.predict(request, model=model)` sends it, and `to_pandas()` returns the result.

`task` is required: it is `'classification'` or `'regression'`, and a column of whole numbers could be either.

**Note:** AdventureWorks is sample data, and the results below are for demonstration.
Benchmark Kumo Tabular on data representative of your own use case before drawing conclusions about its performance.

### Example 1: Predict a product's color

Which color does a product have?

Some products have no color recorded, but only the products that do can be checked.
The cell sets aside a random 60 of those as the rows to score, with their color hidden, and gives the rest as context.
The result is one row per product to score, with the predicted class and a probability for each class.

In [ ]:
TARGET = 'Color'
known = features[features[TARGET].notna()].sample(frac=1, random_state=0)
held_out = known.head(60).reset_index(drop=True)
context = known.iloc[60:]
# Which rows are which, by their place in the table, for Example 3.
held_out_rows, context_rows = known.index[:60], known.index[60:]

request = Request(
    context=context,
    predict=held_out.drop(columns=[TARGET]),
    target=TARGET,
    task='classification',
)
colors = client.predict(request, model=model).to_pandas()
colors.head()

In [ ]:
held_out['predicted'] = held_out.index.map(
    colors.set_index('row_index')['prediction']
)
most_common = context[TARGET].mode()[0]
correct = (held_out['predicted'] == held_out[TARGET]).sum()
guessed = (held_out[TARGET] == most_common).sum()
print(
    f'{correct} of {len(held_out)} colors predicted correctly; always '
    f'guessing the most common color, {most_common}, gets {guessed} right.'
)
display(
    held_out.loc[
        held_out['predicted'] != held_out[TARGET], [TARGET, 'predicted']
    ]
)

How to interpret the result:

1. The first table has one row per product, keyed by `row_index`, its position among the rows to score, with the predicted color in `prediction` and one probability column per color.
2. The printed line counts the correct predictions, and the right guesses of a rule that always picks the most common color.
3. The second table lists the products the model got wrong.

The probabilities are the model's, not calibrated scores: read them as an ordering over the colors rather than as frequencies.

### Example 2: Predict a product's list price

The same table, a number instead of a label.
Changing the question means changing the target and the task; everything else is the same call.
Here the answer is a price, so the task is a regression, and the result has one `prediction` column and no probabilities.

Only products with a list price above zero are used, because a price of zero marks a part that is never sold.
The cell sets aside a random 60 of them as the rows to score, with their price hidden, and compares the predictions with the recorded prices and with a rule that always guesses the median price.
The table also holds each product's standard cost, which tracks its list price closely, so this is an easy question; a column with less to give away is a harder one.

In [ ]:
priced = features[features['ListPrice'] > 0].sample(frac=1, random_state=0)
to_price = priced.head(60).reset_index(drop=True)

request = Request(
    context=priced.iloc[60:],
    predict=to_price.drop(columns=['ListPrice']),
    target='ListPrice',
    task='regression',
)
prices = client.predict(request, model=model).to_pandas()

to_price['predicted'] = to_price.index.map(
    prices.set_index('row_index')['prediction']
)
error = (to_price['predicted'] - to_price['ListPrice']).abs().mean()
median = priced.iloc[60:]['ListPrice'].median()
rule_error = (median - to_price['ListPrice']).abs().mean()
print(
    f'Average error of the predicted price: ${error:,.2f}; always guessing '
    f'the median price, ${median:,.2f}: ${rule_error:,.2f}.'
)
to_price[['ListPrice', 'predicted']].head(10)

How to interpret the result:

1. The printed line compares the average difference between the predicted and recorded price with that of a rule that always guesses the median price.
2. The table shows the first ten products: the recorded price and the predicted one.

### Example 3: Ask the color question in PQL

Example 1 built a `Request` by hand.
The Predictive Query Language says what to predict and which entity it is about, and gives the same question in one line:

```
PREDICT products.Color            ← the column to fill in
FOR EACH products.ProductID       ← one prediction per product
```

The schema comes from the rows already in the notebook, so this asks nothing more of the catalog.
`materialize(predict=...)` takes the rows to score, the same 60 products as Example 1, and builds the request.
`ProductID` is added back here because the schema needs a key.
Almost every other clause of the language, such as aggregations, time windows and `RANK`, is refused on one table, and the refusal says so.

In [ ]:
with_id = products.drop(columns=[c for c in LEFT_OUT if c != 'ProductID'])
ask = with_id.loc[held_out_rows].reset_index(drop=True)
schema = Schema.from_data(
    {'products': with_id.loc[context_rows]},
    primary_keys={'products': 'ProductID'},
)
request = PredictiveQuery(
    f'PREDICT products.{TARGET} FOR EACH products.ProductID', schema
).materialize(predict=ask.drop(columns=[TARGET]), model=model)
pql_colors = client.predict(request, model=model).to_pandas()

ask['predicted'] = ask['ProductID'].map(
    pql_colors.set_index('id')['prediction']
)
print(
    f'{(ask["predicted"] == ask[TARGET]).sum()} of {len(ask)} colors predicted '
    'correctly.'
)
pql_colors.head()

How to interpret the result:

1. The result is keyed by `ProductID`, in the `id` column, as well as by `row_index`.
2. The printed line counts the correct predictions for the same 60 products as Example 1, from the same context rows, so it matches Example 1's count.

## Step 8. Try it yourself

The next cell asks the same price question as Example 2, with a number of context rows, `CONTEXT_ROWS`, that you can change.
It prints the average error of the predicted prices.
Change `CONTEXT_ROWS`, or uncomment the challenge line, and run it again.

In [ ]:
CONTEXT_ROWS = 200
# Challenge: uncomment the line, run the cell again, and compare the error.
# CONTEXT_ROWS = 20  # how few examples can the model learn from?

request = Request(
    context=priced.iloc[60:].head(CONTEXT_ROWS),
    predict=to_price.drop(columns=['ListPrice', 'predicted']),
    target='ListPrice',
    task='regression',
)
result = client.predict(request, model=model).to_pandas()
predicted = to_price.index.map(result.set_index('row_index')['prediction'])
print(
    f'Average error with {CONTEXT_ROWS} context rows: '
    f'${(predicted - to_price["ListPrice"]).abs().mean():,.2f}.'
)

## Step 9. Use your own data

For your own table, name the column to fill in and say which rows to score:

```python
known = df[df['<column>'].notna()]
unknown = df[df['<column>'].isna()].drop(columns=['<column>'])
request = Request(
    context=known,
    predict=unknown,
    target='<column>',
    task='classification',  # or 'regression'
)
result = client.predict(request, model=model).to_pandas()
```

| Question | Target | Task |
|---|---|---|
| Which category does this item belong to? | a categorical column | `classification` |
| Will this customer churn? | a yes-or-no column | `classification` |
| What will this order be worth? | a numerical column | `regression` |
| What is this item's missing weight? | a numerical column | `regression` |

For linked tables and questions about the future, such as who will stop ordering in the next 90 days, use Kumo Relational instead: it reads the links between tables and the time of each event.

Tips:

- Leave out random row IDs, free text, and dates: the model reads numerical and categorical columns.
- A request carries its rows in one call, up to 10,000 context rows and a limit of 16 MB on the request body; send fewer rows if the endpoint refuses it as too large.
- To keep the predictions, save them as a table: `spark.createDataFrame(result).write.saveAsTable('<table>')`.

Learn more:

- [One table](https://github.com/NVIDIA/structured-data-models-client/blob/main/docs/get-started/quickstart.md#one-table) in the client's quickstart
- [Limits](https://github.com/NVIDIA/structured-data-models-client/blob/main/docs/reference/limits.md)

## Step 10. Clean up

The endpoint bills for GPU time only while it runs.
To delete it, uncomment the last line and run the cell.
The sample table stays in your schema until you drop it.

In [ ]:
client.close()
# workspace.serving_endpoints.delete(ENDPOINT_NAME)